
# Phase 5-3: PU-Learning 모델 학습 (XGBoost 버전)
이 노트북은 07번 노트북의 Random Forest 베이스 모델을 **XGBoost**로 교체하여 
동일한 동구(Dong-gu) 지역의 극단적 불균형 데이터를 학습합니다.
훈련된 모델은 `models/universal_xgb_model.pkl`에 저장됩니다.


In [ ]:

import geopandas as gpd
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

base_dir = Path('../../')
dataset_path = base_dir / 'data/processed/dataset_dong-gu_gwangju_south_korea_10m_buf10m_baseline.gpkg'
model_dir = base_dir / 'data/models'
model_dir.mkdir(parents=True, exist_ok=True)

print("✅ 라이브러리 및 경로 설정 완료")


In [ ]:

print(f"📦 동구 데이터셋 로드 중... ({dataset_path.name})")
gdf = gpd.read_file(dataset_path)

# 결측치 처리 (안전장치 - Rule 5 준수)
gdf['is_trash'] = gdf['is_trash'].fillna(0).astype(int)

# 모델에 들어갈 X 피처 컬럼들 자동 추출
feature_cols = [col for col in gdf.columns if '_count_' in col]
X = gdf[feature_cols].values
y = gdf['is_trash'].values

print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:

class PUBaggingXGBoost:
    def __init__(self, n_estimators=10, random_state=42):
        self.n_estimators = n_estimators
        self.models = []
        self.random_state = random_state
        
    def fit(self, X, y):
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        
        np.random.seed(self.random_state)
        
        for i in range(self.n_estimators):
            # Under-sampling
            sampled_u_idx = np.random.choice(u_idx, size=len(p_idx), replace=True)
            
            train_idx = np.concatenate([p_idx, sampled_u_idx])
            X_train = X[train_idx]
            y_train = np.concatenate([np.ones(len(p_idx)), np.zeros(len(sampled_u_idx))])
            
            # XGBoost Classifier 초기화
            model = xgb.XGBClassifier(
                n_estimators=100, 
                max_depth=6, 
                n_jobs=-1, 
                random_state=self.random_state+i, 
                eval_metric='logloss'
            )
            model.fit(X_train, y_train)
            self.models.append(model)
            
    def predict_proba(self, X):
        probas = np.array([model.predict_proba(X)[:, 1] for model in self.models])
        return probas.mean(axis=0)

print("✅ PUBaggingXGBoost 클래스 준비 완료")


In [ ]:

# 모델 초기화 및 학습
model = PUBaggingXGBoost(n_estimators=10, random_state=42)
print("🏋️‍♂️ XGBoost 기반 모델 학습을 시작합니다...")
model.fit(X, y)
print("✨ 모델 학습 완료!")

# 결과 텐서 검증 (Rule 5)
sample_scores = model.predict_proba(X[:1000])
# Rule 5: 점수 정규화 및 결측치 방어 강제화 테스트
sample_scores = np.nan_to_num(sample_scores, nan=0.0)
sample_scores = np.clip(sample_scores, 0.0, 1.0)

print(f"샘플 스코어 최소값: {sample_scores.min():.4f}, 최대값: {sample_scores.max():.4f}")
if sample_scores.min() >= 0.0 and sample_scores.max() <= 1.0:
    print("🟩 [PASS] Rule 5 통과. 안전한 점수 분포입니다.")

# 모델 저장
model_save_path = model_dir / 'universal_xgb_model.pkl'
joblib.dump(model, model_save_path)
print(f"🚀 [완료] XGBoost 모델 저장 성공! -> {model_save_path}")
